Using Viterbi Algorithm to find log probability by calculating transition and emission matrices

In [8]:
import math

# States
states = ['E', 'I']

# Transition probabilities (log space)
transition_probs = {
    'E': {'E': math.log(0.9), 'I': math.log(0.1)},
    'I': {'E': math.log(0.1), 'I': math.log(0.9)}
}

# Emission probabilities (log space)
emission_probs = {
    'E': {'A': math.log(0.25), 'C': math.log(0.25), 'G': math.log(0.25), 'T': math.log(0.25)},
    'I': {'A': math.log(0.4),  'C': math.log(0.1),  'G': math.log(0.1),  'T': math.log(0.4)}
}

# Start probabilities (assume we always start in Exon)
start_probs = {'E': 0.0, 'I': float('-inf')}  # log(1) = 0, log(0) = -inf

In [15]:
def get_log_prob_of_a_given_path(state_path, observed_sequence):
    log_prob = start_probs[state_path[0]] * emission_probs[state_path[0]][observed_sequence[0]]
    
    for i in range(1, len(state_path)):
        prev_state = state_path[i - 1]
        curr_state = state_path[i]
        obs = observed_sequence[i]
        
        log_prob += transition_probs[prev_state][curr_state]
        log_prob += emission_probs[curr_state][obs]
    
    return round(log_prob, 2)

In [ ]:
state_path = "EEEEEEEEEEEEEEEEEEEEEEEIIIII" 
obs_sequence = "CTTCATGTTGGCCTTTGAAAGCAGACGT"

log_prob = get_log_prob_of_a_given_path(state_path, obs_sequence)
print("Log-prob of given path:", log_prob) 

Log-prob of given path: -44.28


In [5]:
def viterbi(obs_seq, states, start_p, trans_p, emit_p):
    n = len(obs_seq)
    V = [{} for _ in range(n)]  # Viterbi table
    path = {}

    # Initialization
    for state in states:
        V[0][state] = start_p[state] + emit_p[state][obs_seq[0]]
        path[state] = [state]

    # Recursion
    for t in range(1, n):
        new_path = {}
        for curr_state in states:
            (max_prob, prev_st) = max(
                (V[t-1][prev_state] + trans_p[prev_state][curr_state] + emit_p[curr_state][obs_seq[t]], prev_state)
                for prev_state in states
            )
            V[t][curr_state] = max_prob
            new_path[curr_state] = path[prev_st] + [curr_state]
        path = new_path

    # Termination
    (final_prob, final_state) = max((V[n-1][state], state) for state in states)
    return final_prob, ''.join(path[final_state])

In [6]:
obs = "CTTCATGTGAAAGCAGACGT"
log_p, viterbi_path = viterbi(obs, states, start_probs, transition_probs, emission_probs)
print("Most likely path:", viterbi_path)
print("Log-prob of path:", round(log_p, 2))

Most likely path: EEEEEEEEEEEEEEEEEEEE
Log-prob of path: -29.73
